# 2. GitHub Actions OIDC con Vault

Este notebook configura autenticación OIDC de GitHub Actions contra Vault usando `vault` CLI para el lado de Vault y `gh` CLI para el lado de GitHub.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('/tmp/vault/config.env')

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

In [ ]:
WORKDIR = '/tmp/vault'
REPO_FULL_NAME = 'jm-merchan/Mapfre_PoC_Vault'
REPO_OWNER, REPO_NAME = REPO_FULL_NAME.split('/', 1)
GHA_BRANCH = 'main'
VAULT_JWT_PATH = 'github'
VAULT_POLICY_NAME = f'gha-{REPO_NAME}'
VAULT_ROLE_NAME = f'{VAULT_POLICY_NAME}-{GHA_BRANCH}'

os.environ.update({
    'WORKDIR': WORKDIR,
    'REPO_FULL_NAME': REPO_FULL_NAME,
    'REPO_OWNER': REPO_OWNER,
    'REPO_NAME': REPO_NAME,
    'GHA_BRANCH': GHA_BRANCH,
    'VAULT_JWT_PATH': VAULT_JWT_PATH,
    'VAULT_POLICY_NAME': VAULT_POLICY_NAME,
    'VAULT_ROLE_NAME': VAULT_ROLE_NAME,
})
os.environ.pop('VAULT_SKIP_VERIFY', None)
os.makedirs(f'{WORKDIR}/gha', exist_ok=True)

## Policy de Vault para GitHub Actions

La policy de ejemplo permite leer secretos en `secret/data/gha/*` (KV v2). Ajusta paths/capabilities según tu caso.

In [ ]:
%%bash
set -euo pipefail

vault status
cat > ${WORKDIR}/gha/${VAULT_POLICY_NAME}.hcl <<EOF
path "secret/data/gha/*" {
  capabilities = ["read"]
}

path "secret/metadata/gha/*" {
  capabilities = ["read", "list"]
}
EOF

vault policy write ${VAULT_POLICY_NAME} ${WORKDIR}/gha/${VAULT_POLICY_NAME}.hcl
vault policy read ${VAULT_POLICY_NAME}

In [ ]:
%%bash
set -euo pipefail
if ! vault auth list -format=json | jq -e --arg path "${VAULT_JWT_PATH}/" 'has($path)' >/dev/null; then
  vault auth enable -path=${VAULT_JWT_PATH} jwt
fi

vault write auth/${VAULT_JWT_PATH}/config \
  oidc_discovery_url="https://token.actions.githubusercontent.com" \
  bound_issuer="https://token.actions.githubusercontent.com"

vault read auth/${VAULT_JWT_PATH}/config

In [ ]:
%%bash
set -euo pipefail
cat > ${WORKDIR}/gha/role-${VAULT_ROLE_NAME}.json <<EOF
{
  "role_type": "jwt",
  "user_claim": "actor",
  "bound_audiences": "https://github.com/${REPO_OWNER}",
  "bound_claims_type": "glob",
  "bound_claims": {
    "repository": "${REPO_FULL_NAME}",
    "ref": "refs/heads/${GHA_BRANCH}"
  },
  "token_policies": "${VAULT_POLICY_NAME}",
  "token_ttl": "1h"
}
EOF

vault write auth/${VAULT_JWT_PATH}/role/${VAULT_ROLE_NAME} @${WORKDIR}/gha/role-${VAULT_ROLE_NAME}.json
vault read auth/${VAULT_JWT_PATH}/role/${VAULT_ROLE_NAME}

## Workflow de ejemplo en GitHub Actions

Este job solicita un token OIDC (`id-token: write`), autentica contra Vault y lee un secreto.

Requisitos en el repositorio de GitHub:
- Variable `VAULT_ADDR` (URL de Vault, por ejemplo `https://...`)
- Variable `VAULT_AUTH_PATH` (en este ejemplo `github`)
- Variable `VAULT_AUTH_ROLE` (rol creado en este notebook)

In [ ]:
%%bash
set -euo pipefail

WORKFLOW_FILE=.github/workflows/vault-oidc.yml
mkdir -p .github/workflows
cat > ${WORKFLOW_FILE} <<'EOF'
name: vault-oidc
on:
  workflow_dispatch:
jobs:
  read-secret:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      id-token: write
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Import Vault Secrets
        id: vault
        uses: hashicorp/vault-action@v3
        with:
          url: ${{ vars.VAULT_ADDR }}
          tlsSkipVerify: true.               # Requerido puesto que estamos usando un self-signed certificate en el demo.
          method: jwt
          path: ${{ vars.VAULT_AUTH_PATH }}
          role: ${{ vars.VAULT_AUTH_ROLE }}
          secrets: |
            secret/data/gha/demo api_key | API_KEY

      - name: Use Secret
        run: |
          test -n "${API_KEY}"
          echo "Secret loaded successfully"
EOF

gh variable set VAULT_AUTH_PATH --repo "${REPO_FULL_NAME}" --body "${VAULT_JWT_PATH}"
gh variable set VAULT_AUTH_ROLE --repo "${REPO_FULL_NAME}" --body "${VAULT_ROLE_NAME}"

if [ -n "${VAULT_ADDR:-}" ]; then
  gh variable set VAULT_ADDR --repo "${REPO_FULL_NAME}" --body "${VAULT_ADDR}"
fi

echo "Workflow generado localmente en ${WORKFLOW_FILE}"
echo "Sube el workflow al repo ${REPO_FULL_NAME} con git push o gh api repos/.../contents si prefieres API."
echo "Variables GHA configuradas: VAULT_AUTH_PATH, VAULT_AUTH_ROLE${VAULT_ADDR:+, VAULT_ADDR}"

In [ ]:
%%bash
set -euo pipefail

WORKFLOW_FILE=.github/workflows/vault-oidc.yml
: "${REPO_FULL_NAME:?Ejecuta primero las celdas 2 y 3.}"
test -f "${WORKFLOW_FILE}"

WORKFLOW_SHA="$(gh api -q .sha "repos/${REPO_FULL_NAME}/contents/${WORKFLOW_FILE}" 2>/dev/null || true)"
CONTENT="$(base64 < "${WORKFLOW_FILE}" | tr -d '\n')"
ARGS=(--method PUT "repos/${REPO_FULL_NAME}/contents/${WORKFLOW_FILE}" -f message="Add Vault OIDC workflow" -f content="${CONTENT}")

if [ -n "${WORKFLOW_SHA}" ]; then
  ARGS+=(-f sha="${WORKFLOW_SHA}")
fi

gh api "${ARGS[@]}" --jq .content.path